# Gold Layer: Business Analytics and Aggregations

This notebook creates business-ready aggregations from the silver layer for analytical queries and reporting.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, sum, avg, max, min, stddev, round, desc,
    current_timestamp, lit
)
from datetime import datetime

print("=" * 80)
print("GOLD LAYER AGGREGATION: Amazon Products Analytics")
print("=" * 80)
print(f"Start Time: {datetime.now()}")
print()

In [ ]:
# Configuration
source_table = "silver_amazon_products"
target_table = "gold_amazon_products"

print(f"Source: {source_table}")
print(f"Target: {target_table}")
print()

In [ ]:
# Read from Silver Layer
print(f"Reading from silver layer: {source_table}")
try:
    df_silver = spark.table(source_table)
    silver_count = df_silver.count()
    print(f"✅ Read {silver_count} rows from silver layer")
    print()
except Exception as e:
    print(f"❌ Error reading from silver layer: {str(e)}")
    raise

In [ ]:
# Business Aggregations
print("Creating business aggregations...")

# Aggregation 1: Product Statistics by Category
if "category" in df_silver.columns and "price" in df_silver.columns:
    df_category_stats = df_silver \
        .groupBy("category") \
        .agg(
            count("*").alias("product_count"),
            round(avg("price"), 2).alias("avg_price"),
            round(max("price"), 2).alias("max_price"),
            round(min("price"), 2).alias("min_price"),
            round(stddev("price"), 2).alias("stddev_price")
        ) \
        .orderBy(desc("product_count"))

    print("Category Statistics:")
    print("-" * 80)
    df_category_stats.show(10, truncate=False)
    print()
else:
    # If category or price columns don't exist, create a simple aggregation
    df_category_stats = df_silver \
        .agg(count("*").alias("product_count")) \
        .withColumn("category", lit("ALL"))

In [ ]:
# Aggregation 2: Overall Product Summary
df_summary = df_silver \
    .agg(
        count("*").alias("total_products"),
        round(avg("price"), 2).alias("overall_avg_price") if "price" in df_silver.columns else lit(0).alias("overall_avg_price")
    ) \
    .withColumn("report_timestamp", current_timestamp()) \
    .withColumn("report_type", lit("product_summary"))

print("Overall Summary:")
print("-" * 80)
df_summary.show(truncate=False)
print()

In [ ]:
# Prepare Gold Layer Output
print("Preparing gold layer output...")

# For this example, we'll use the category stats as the main gold layer
df_gold = df_category_stats \
    .withColumn("gold_processing_timestamp", current_timestamp()) \
    .withColumn("aggregation_level", lit("category")) \
    .withColumn("gold_layer_version", lit("1.0"))

gold_count = df_gold.count()
print(f"✅ Created {gold_count} aggregated rows for gold layer")
print()

In [ ]:
# Display Gold Layer Data
print("Gold Layer Schema:")
print("-" * 80)
df_gold.printSchema()
print()

print("Gold Layer Data:")
print("-" * 80)
df_gold.show(20, truncate=False)
print()

In [ ]:
# Write to Gold Layer
print(f"Writing to gold layer: {target_table}")
try:
    df_gold.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)

    print(f"✅ Successfully wrote {gold_count} rows to {target_table}")
    print()

except Exception as e:
    print(f"❌ Error writing to gold layer: {str(e)}")
    raise

In [ ]:
# Verification
print("Verifying data in gold layer...")
df_verify = spark.table(target_table)
verify_count = df_verify.count()
print(f"✅ Verified: {verify_count} rows in {target_table}")
print()

In [ ]:
# Summary
print("=" * 80)
print("GOLD LAYER AGGREGATION COMPLETE")
print("=" * 80)
print(f"Source Table: {source_table}")
print(f"Target Table: {target_table}")
print(f"Input Rows: {silver_count}")
print(f"Aggregated Rows: {gold_count}")
print(f"Aggregation Type: Category-level statistics")
print(f"End Time: {datetime.now()}")
print()